In [1]:
!pip install -U transformers
!pip install accelerate -q

In [2]:
%%capture
!pip install --no-deps --upgrade timm # Only for Gemma 3N

In [3]:
!pip install opencv-python

In [4]:
!pip uninstall torchcodec -y

In [ ]:
from huggingface_hub import login
import os

# Paste your token here (get it from: https://huggingface.co/settings/tokens)
HF_TOKEN = "" 

login(token=HF_TOKEN)

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch

# Load the 2B finetuned model using transformers directly
processor = AutoProcessor.from_pretrained("blind-assist/gemma-3n-2b-finetune-e3-8500")
model = AutoModelForImageTextToText.from_pretrained(
    "blind-assist/gemma-3n-2b-finetune-e3-8500",
    torch_dtype=torch.bfloat16,
    device_map="cuda"
)

print(f"Model loaded on: {model.device}")

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1556 [00:00<?, ?it/s]

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



Model loaded on: cuda:0


In [7]:
import cv2
import numpy as np
from PIL import Image

def downsample_video(video_path, num_frames=2):
    """Extracts evenly spaced frames for VLM context."""
    
    vidcap = cv2.VideoCapture(video_path)
    total_frames = int(vidcap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0: return []
    
    indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)
    frames = []
    for i in indices:
        vidcap.set(cv2.CAP_PROP_POS_FRAMES, i)
        success, image = vidcap.read()
        if success:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            frames.append(Image.fromarray(image))
    vidcap.release()
    return frames

In [ ]:
import os
import time
import csv
from pathlib import Path

# Setup paths
video_folder = "videos"
output_csv = "video_guidance_results_2b.csv"

# Get all video files
video_files = sorted([f for f in os.listdir(video_folder) 
                     if f.endswith(('.mp4', '.avi', '.mov', '.mkv'))])

print(f"Found {len(video_files)} videos to process")

# Instruction for the model
instruction = ("Given the visual input from the user's forward perspective, "
               "generate exactly one short sentence to guide a visually impaired user "
               "by identifying critical obstacles or landmarks, describing their locations "
               "using clock directions relative to the user (12 o'clock is straight ahead), "
               "including relevant details such as size, material, or distance, and giving "
               "one clear action, while prioritizing immediate safety and avoiding any extra explanation.")

# Prepare CSV file
results = []

# Process each video
for idx, video_file in enumerate(video_files, 1):
    video_path = os.path.join(video_folder, video_file)
    print(f"\n[{idx}/{len(video_files)}] Processing: {video_file}")
    
    try:
        # Extract frames from video
        frames = downsample_video(video_path)
        
        if not frames:
            raise ValueError("No frames extracted from video")
        
        # Build the message with images and text
        content = []
        for img in frames:
            content.append({"type": "image", "image": img})
        content.append({"type": "text", "text": instruction})
        
        messages = [{"role": "user", "content": content}]
        
        # Process inputs using apply_chat_template (this works for the finetuned model)
        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to(model.device)

        print(f"Generating guidance for {video_file}...")
        
        start_time = time.time()
        
        output = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            # use_cache=True,
            # temperature=1.0,
            # top_p=0.95,
            # top_k=64
        )

        end_time = time.time()
        inference_time = round(end_time - start_time, 3)
        
        # Decode only the generated tokens (excluding the input)
        generated_text = processor.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
        
        # Clean up the output
        guidance = " ".join(generated_text.strip().split())
        
        print(f"Output: {guidance}")
        
        # Store result
        results.append({
            "video_id": idx,
            "video_filename": video_file,
            "guidance": guidance,
            "inference_time_s": inference_time
        })
        
    except Exception as e:
        import traceback
        print(f"Error processing {video_file}: {str(e)}")
        traceback.print_exc()
        results.append({
            "video_id": idx,
            "video_filename": video_file,
            "guidance": f"ERROR: {str(e)}",
            "inference_time_s": 0.0
        })

# Save results to CSV
print(f"\nSaving results to {output_csv}...")
with open(output_csv, 'w', newline='', encoding='utf-8') as csvfile:
    fieldnames = ['video_id', 'video_filename', 'guidance', 'inference_time_s']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    
    writer.writeheader()
    writer.writerows(results)

print(f"\nProcessing complete! Results saved to {output_csv}")
print(f"Successfully processed {len([r for r in results if not r['guidance'].startswith('ERROR')])} out of {len(video_files)} videos")

Found 15 videos to process

[1/15] Processing: 20240914_1abe4d6f616ccb2d8e15049136e1656d_4m28s.mp4
Generating guidance for 20240914_1abe4d6f616ccb2d8e15049136e1656d_4m28s.mp4...
Output: There are pedestrians passing at 11 o'clock direction and 5 steps away. Please keep slow.

[2/15] Processing: 20240918-youtube_short_081e0a96bac802b988a1db9df310ddd1_1min03s.mp4
Generating guidance for 20240918-youtube_short_081e0a96bac802b988a1db9df310ddd1_1min03s.mp4...
Output: There are passers - by staying at one o'clock direction. There is a small vendor about five steps away at eleven o'clock direction. The road is narrow, so slow down.

[3/15] Processing: 20240918-youtube_short_11bc0f3b2d7f68e62f5d61a10d2f8897_4min03s.mp4
Generating guidance for 20240918-youtube_short_11bc0f3b2d7f68e62f5d61a10d2f8897_4min03s.mp4...
Output: at 11 o'clock there is a railing, be careful to avoid it.

[4/15] Processing: 20240918-youtube_short_1aec7c3e80c13cbb75f75f72333148bf_2min24s.mp4
Generating guidance for 202409

In [9]:
# Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA A40. Max memory = 44.339 GB.
10.268 GB of memory reserved.
